# Saliency by shaping in tree-based shape space

This tutorial follows the official Higra shaping example using this project's `ShapeSpaceSaliency` API. The default uses a Tree of Shapes, circularity, maxima in shape space, and `skimage.data.coins()`. Primary reference: Yongchao Xu, Edwin Carlinet, Thierry Géraud, and Laurent Najman, [Hierarchical Segmentation Using Tree-Based Shape Spaces](https://doi.org/10.1109/TPAMI.2016.2554550), *IEEE Transactions on Pattern Analysis and Machine Intelligence* 39(3), 457–469, 2017. The library's maxima and edge-indexed generalizations are documented in [`docs/saliency.md`](../docs/saliency.md#primary-references-and-implementation-correspondence).

## Goal

The pipeline:

1. build an original morphological hierarchy `T`;
2. use its nodes as graph vertices and parent-child relations as graph edges;
3. compute an arbitrary node attribute `A`;
4. find regional maxima or minima of `A` in shape space and compute extinction;
5. store extinction on each canonical representative;
6. assign every image edge the maximum score among region contours containing it.

This is not `valuation(LCA(owner(p), owner(q)))`, which belongs to `HierarchySaliencyMap`.

## Setup

The notebook imports the installed package directly and obtains its image from scikit-image. Prepare dependencies outside the notebook by following `README.md`.

In [ ]:
from __future__ import annotations

import mmcfilters
from IPython.display import display
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage import data

%matplotlib inline
plt.rcParams.update({"figure.dpi": 105, "axes.titlesize": 11, "axes.labelsize": 10})
print(f"mmcfilters: {mmcfilters.__version__}")


### Parameters

Change this cell to test another tree or criterion. `MIN_AREA` controls texture simplification. `TOP_K=24` approximately matches the visible coins. `Maxima` is appropriate when larger attribute values represent stronger shapes; use `Minima` for the inverse case.

In [ ]:
IMAGE_SOURCE = "skimage.data.coins()"

TREE_KIND = "Tree of Shapes"   # "Tree of Shapes", "Max-tree", or "Min-tree"
ATTRIBUTE_NAME = "CIRCULARITY"   # COMPACTNESS, CIRCULARITY, RECTANGULARITY, or BITQUADS_CIRCULARITY
POLARITY_NAME = "Maxima"   # "Maxima" or "Minima"
MIN_AREA = 500  # remove structures smaller than this pixel count

TREE_RADIUS = 1.0
CONTOUR_RADIUS = 1.0
TOP_K = 24  # one dominant shape per visible coin
CUT_QUANTILES = (0.80, 0.90, 0.97)


image = np.ascontiguousarray(data.coins(), dtype=np.uint8)
print(
    f"image: {IMAGE_SOURCE}, shape={image.shape}, dtype={image.dtype}, "
    f"range=[{int(image.min())}, {int(image.max())}]"
)

plt.figure(figsize=(7.0, 5.5))
plt.imshow(image, cmap="gray")
plt.title("Input image: scikit-image coins")
plt.axis("off")
plt.show()

## Steps

### 1. Build the original hierarchy

The Tree of Shapes is closest to the reference example. Components below `MIN_AREA` are merged into their parent before shaping, reducing internal coin texture. Max-tree and min-tree alternatives are also available. Final contour adjacency is explicit.

In [ ]:
def build_tree(kind: str, image_uint8: np.ndarray):
    if kind == "Tree of Shapes":
        return mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(
            image_uint8,
            interpolation=mmcfilters.ToSInterpolation.SelfDual,
        )
    if kind == "Max-tree":
        return mmcfilters.MorphologicalTreeFactory.createMaxTree(
            image_uint8,
            radius=TREE_RADIUS,
        )
    if kind == "Min-tree":
        return mmcfilters.MorphologicalTreeFactory.createMinTree(
            image_uint8,
            radius=TREE_RADIUS,
        )
    raise ValueError(f"unknown tree: {kind}")


def merge_nodes_below_area(tree_to_simplify, min_area: int) -> int:
    area = mmcfilters.Attribute.computeSingleTopologyAttribute(
        tree_to_simplify,
        mmcfilters.Attribute.AREA,
    )
    root_id = int(tree_to_simplify.root)
    removed_nodes = 0
    for node_id in tree_to_simplify.getPostOrderNodes():
        node_id = int(node_id)
        if (
            node_id != root_id
            and tree_to_simplify.isAlive(node_id)
            and area[node_id] < min_area
        ):
            tree_to_simplify.mergeNodeIntoParent(node_id)
            removed_nodes += 1
    return removed_nodes


tree = build_tree(TREE_KIND, image)
nodes_before_simplification = len(tree.aliveNodeIds)
removed_nodes = merge_nodes_below_area(tree, MIN_AREA)
alive_nodes = np.asarray(tree.aliveNodeIds, dtype=np.int64)

print(f"tree: {TREE_KIND}")
print(f"internal slots: {tree.numInternalNodeSlots}")
print(f"live nodes before simplification: {nodes_before_simplification}")
print(f"nodes merged by area < {MIN_AREA}: {removed_nodes}")
print(f"live nodes after simplification: {alive_nodes.size}")
print(f"adjacency mode: {tree.adjacencyMode}")
print(f"uniform adjacency: {tree.hasUniformGridAdjacency2D}")
print(f"directional adjacency: {tree.hasDirectionalGridAdjacency2D}")

### 2. Compute an arbitrary attribute

The attribute need not be monotone on the simplified original hierarchy; that is precisely why a second hierarchy is built in shape space.

In [ ]:
ATTRIBUTE_OPTIONS = {
    "COMPACTNESS": mmcfilters.Attribute.COMPACTNESS,
    "CIRCULARITY": mmcfilters.Attribute.CIRCULARITY,
    "RECTANGULARITY": mmcfilters.Attribute.RECTANGULARITY,
    "BITQUADS_CIRCULARITY": mmcfilters.Attribute.BITQUADS_CIRCULARITY,
}
POLARITY_OPTIONS = {
    "Maxima": mmcfilters.ShapeSpaceExtremaPolarity.Maxima,
    "Minima": mmcfilters.ShapeSpaceExtremaPolarity.Minima,
}

node_attribute = mmcfilters.Attribute.computeSingleTopologyAttribute(
    tree,
    ATTRIBUTE_OPTIONS[ATTRIBUTE_NAME],
).astype(np.float32, copy=False)
node_attribute = np.ascontiguousarray(node_attribute)
polarity = POLARITY_OPTIONS[POLARITY_NAME]

live_attribute = node_attribute[alive_nodes]
print(
    f"attribute: {ATTRIBUTE_NAME}, dtype={node_attribute.dtype}, "
    f"live range=[{float(live_attribute.min()):.6g}, "
    f"{float(live_attribute.max()):.6g}]"
)

### 3. Compute extinction and accumulate it on contours

The API exposes extinction and projection as separate stages and provides `compute` for the complete pipeline.

In [ ]:
extinction_result = mmcfilters.ShapeSpaceSaliency.computeExtinctionValues(
    tree,
    attribute=node_attribute,
    polarity=polarity,
)

staged_edge_map = mmcfilters.ShapeSpaceSaliency.projectContourScores(
    tree,
    extinction_result["nodeScores"],
    radius=CONTOUR_RADIUS,
)

shape_space_result = mmcfilters.ShapeSpaceSaliency.compute(
    tree,
    attribute=node_attribute,
    polarity=polarity,
    radius=CONTOUR_RADIUS,
)
edge_map = shape_space_result["edgeMap"]

np.testing.assert_array_equal(
    shape_space_result["nodeScores"],
    extinction_result["nodeScores"],
)
for key in ("sources", "targets", "values"):
    np.testing.assert_array_equal(edge_map[key], staged_edge_map[key])

edge_values = np.asarray(edge_map["values"])
node_scores = np.asarray(shape_space_result["nodeScores"])

print(f"regional extrema: {len(shape_space_result['extrema'])}")
print(f"representatives with positive extinction: {np.count_nonzero(node_scores)}")
print(f"image edges: {edge_values.size}")
print(f"edges with positive saliency: {np.count_nonzero(edge_values)}")
print(f"maximum saliency: {float(edge_values.max()):.6g}")

### 4. Inspect the strongest extrema

`birthLevel` and `deathLevel` stay in the original attribute domain. Maxima use `birthLevel - deathLevel`; minima use `deathLevel - birthLevel`.

In [ ]:
extrema_frame = pd.DataFrame(shape_space_result["extrema"])
if extrema_frame.empty:
    strongest_extrema = extrema_frame
else:
    strongest_extrema = (
        extrema_frame
        .sort_values(
            ["extinction", "representative"],
            ascending=[False, True],
        )
        .head(12)
        .reset_index(drop=True)
    )

display(strongest_extrema)

### 5. Keep only the strongest extrema

Because extinction and projection are separate, retain the `TOP_K` largest representative scores and project the sparse vector again.

In [ ]:
ranked_extrema = sorted(
    shape_space_result["extrema"],
    key=lambda item: (
        -float(item["extinction"]),
        int(item["representative"]),
    ),
)
selected_representatives = np.asarray(
    [
        int(item["representative"])
        for item in ranked_extrema[:TOP_K]
        if float(item["extinction"]) > 0
    ],
    dtype=np.int64,
)

selected_node_scores = np.zeros_like(node_scores)
selected_node_scores[selected_representatives] = node_scores[selected_representatives]
selected_edge_map = mmcfilters.ShapeSpaceSaliency.projectContourScores(
    tree,
    selected_node_scores,
    radius=CONTOUR_RADIUS,
)
selected_edge_values = np.asarray(selected_edge_map["values"])

print(f"retained extrema: {selected_representatives.size} / {len(ranked_extrema)}")
print(f"salient edges after top-{TOP_K}: {np.count_nonzero(selected_edge_values)}")

## Results

Figures show the attribute distribution, extinction, and top-k selection. Images are display rasterizations; the canonical result remains the edge-indexed dictionary returned by `ShapeSpaceSaliency`.

In [ ]:
def edge_map_to_pixel_image(edge_map_dict: dict) -> np.ndarray:
    return np.asarray(
        mmcfilters.HierarchySaliencyMapProjection.edgeMapToPixelImage(
            edge_map_dict,
            mmcfilters.EdgeToPixelReducer.Max,
        )
    )


def edge_segments(edge_container: dict) -> np.ndarray:
    cols = int(edge_container["numCols"])
    sources = np.asarray(edge_container["sources"], dtype=np.int64)
    targets = np.asarray(edge_container["targets"], dtype=np.int64)
    source_y, source_x = np.divmod(sources, cols)
    target_y, target_x = np.divmod(targets, cols)
    return np.stack(
        [
            np.column_stack([source_x, source_y]),
            np.column_stack([target_x, target_y]),
        ],
        axis=1,
    )


def plot_contour_cuts(
    image_uint8: np.ndarray,
    cuts: list[tuple[float, dict]],
) -> None:
    fig, axes = plt.subplots(
        1,
        len(cuts),
        figsize=(4.4 * len(cuts), 4.2),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)
    for ax, (threshold, cut) in zip(axes, cuts):
        ax.imshow(image_uint8, cmap="gray")
        segments = edge_segments(cut)
        if segments.size:
            ax.add_collection(
                LineCollection(
                    segments,
                    colors="crimson",
                    linewidths=0.55,
                    alpha=0.85,
                )
            )
        ax.set_xlim(-0.5, image_uint8.shape[1] - 0.5)
        ax.set_ylim(image_uint8.shape[0] - 0.5, -0.5)
        ax.set_aspect("equal")
        ax.set_title(
            f"λ ≥ {threshold:.4g}\n"
            f"{len(cut['sources'])} edges"
        )
        ax.axis("off")
    plt.show()

In [ ]:
positive_extinctions = node_scores[node_scores > 0]
saliency_image = edge_map_to_pixel_image(edge_map)
selected_saliency_image = edge_map_to_pixel_image(selected_edge_map)
common_max = float(max(edge_values.max(), selected_edge_values.max()))

selected_mask = selected_edge_values > 0
selected_positive_edges = dict(selected_edge_map)
for key in ("sources", "targets", "values"):
    selected_positive_edges[key] = np.asarray(selected_edge_map[key])[selected_mask]

fig, axes = plt.subplots(2, 3, figsize=(15, 7), constrained_layout=True)

axes[0, 0].imshow(image, cmap="gray")
axes[0, 0].set_title("Image")
axes[0, 0].axis("off")

axes[0, 1].hist(live_attribute, bins=60, color="steelblue")
axes[0, 1].set_title(f"Distribution: {ATTRIBUTE_NAME}")
axes[0, 1].set_xlabel("attribute value")
axes[0, 1].set_ylabel("nodes")

axes[0, 2].hist(positive_extinctions, bins=60, color="darkorange")
axes[0, 2].set_title("Positive extinctions")
axes[0, 2].set_xlabel("extinction")
axes[0, 2].set_ylabel("representatives")

full_plot = axes[1, 0].imshow(
    saliency_image,
    cmap="magma",
    vmin=0,
    vmax=common_max,
)
axes[1, 0].set_title("All extrema")
axes[1, 0].axis("off")
fig.colorbar(full_plot, ax=axes[1, 0], fraction=0.046, pad=0.04)

selected_plot = axes[1, 1].imshow(
    selected_saliency_image,
    cmap="magma",
    vmin=0,
    vmax=common_max,
)
axes[1, 1].set_title(f"Top-{TOP_K}")
axes[1, 1].axis("off")
fig.colorbar(selected_plot, ax=axes[1, 1], fraction=0.046, pad=0.04)

axes[1, 2].imshow(image, cmap="gray")
selected_segments = edge_segments(selected_positive_edges)
if selected_segments.size:
    axes[1, 2].add_collection(
        LineCollection(
            selected_segments,
            colors="crimson",
            linewidths=0.45,
            alpha=0.9,
        )
    )
axes[1, 2].set_xlim(-0.5, image.shape[1] - 0.5)
axes[1, 2].set_ylim(image.shape[0] - 0.5, -0.5)
axes[1, 2].set_aspect("equal")
axes[1, 2].set_title(f"Contours of top-{TOP_K}")
axes[1, 2].axis("off")

plt.show()

### 6. Cut the saliency hierarchy

Edges with saliency at least `lambda` form nested contours. Quantiles of strictly positive values provide attribute-adaptive thresholds.

In [ ]:
positive_edge_values = edge_values[edge_values > 0]
if positive_edge_values.size == 0:
    raise RuntimeError("The selected configuration produced no positive-saliency edges")

cut_thresholds = np.unique(
    np.quantile(positive_edge_values, CUT_QUANTILES)
)
cuts = [
    (
        float(threshold),
        mmcfilters.HierarchySaliencyMapProjection.thresholdCut(
            edge_map,
            float(threshold),
        ),
    )
    for threshold in cut_thresholds
]

cut_summary = pd.DataFrame(
    {
        "threshold": [threshold for threshold, _ in cuts],
        "edges": [len(cut["sources"]) for _, cut in cuts],
        "fraction_of_all_edges": [
            len(cut["sources"]) / edge_values.size
            for _, cut in cuts
        ],
    }
)
display(cut_summary)
plot_contour_cuts(image, cuts)

## Checks

The checks cover staged/combined equivalence, domain, finiteness, non-negativity, and the fact that circularity changes in both directions along parent-child relations. The latter confirms it is not a monotone valuation of the original hierarchy.

In [ ]:
assert node_attribute.shape == (tree.numInternalNodeSlots,)
assert node_scores.shape == (tree.numInternalNodeSlots,)
assert selected_node_scores.shape == (tree.numInternalNodeSlots,)
assert np.all(np.isfinite(node_attribute[alive_nodes]))
assert np.all(np.isfinite(node_scores[alive_nodes]))
assert np.all(node_scores[alive_nodes] >= 0)
assert np.all(np.isfinite(edge_values))
assert np.all(edge_values >= 0)
assert np.all(np.isfinite(selected_edge_values))
assert np.all(selected_edge_values >= 0)
assert np.all(selected_edge_values <= edge_values)
assert np.count_nonzero(selected_node_scores) <= TOP_K

if np.isclose(CONTOUR_RADIUS, 1.0):
    rows, cols = image.shape
    expected_grid_edges = rows * (cols - 1) + (rows - 1) * cols
    assert edge_values.size == expected_grid_edges

increases_to_parent = 0
decreases_to_parent = 0
ties_to_parent = 0
root = int(tree.root)
for node_id in alive_nodes:
    node_id = int(node_id)
    if node_id == root:
        continue
    parent_id = int(tree.getNodeParent(node_id))
    parent_value = node_attribute[parent_id]
    node_value = node_attribute[node_id]
    if parent_value > node_value:
        increases_to_parent += 1
    elif parent_value < node_value:
        decreases_to_parent += 1
    else:
        ties_to_parent += 1

representatives = [
    int(item["representative"])
    for item in shape_space_result["extrema"]
]
assert len(representatives) == len(set(representatives))
assert set(representatives).issubset(set(map(int, alive_nodes)))
assert set(map(int, selected_representatives)).issubset(set(representatives))

print("numerical checks: OK")
print(f"attribute increases toward parent: {increases_to_parent}")
print(f"attribute decreases toward parent: {decreases_to_parent}")
print(f"ties: {ties_to_parent}")
if increases_to_parent and decreases_to_parent:
    print(
        "The attribute is not monotone on the original tree; "
        "ShapeSpaceSaliency is the appropriate API for this pipeline."
    )

### Conceptual check: shaping is not LCA projection

For the chain `B subset A subset Omega`, attribute `[1, 4, 10]` has a dominant minimum at `B` with extinction 9. Maximum-on-contours produces `[9, 0, 0]`; monotone valuation `[9, 9, 9]` projected by LCA produces `[9, 9, 0]`.

In [ ]:
toy_parent = [4, 5, 6, 6, 5, 6, 6]
toy_altitude = np.array([3, 2, 1, 1, 3, 2, 1], dtype=np.uint8)
toy_tree = mmcfilters.MorphologicalTreeFactory.createFromHigraParent(
    toy_parent,
    toy_altitude,
    1,
    4,
    mmcfilters.MorphologicalTreeKind.MAX_TREE,
    radius=1.0,
)

toy_attribute = np.array([1.0, 4.0, 10.0], dtype=np.float32)
toy_shaping = mmcfilters.ShapeSpaceSaliency.compute(
    toy_tree,
    toy_attribute,
    mmcfilters.ShapeSpaceExtremaPolarity.Minima,
)

toy_lca_valuation = np.array([9.0, 9.0, 9.0], dtype=np.float32)
toy_lca = mmcfilters.HierarchySaliencyMap.computeSaliencyEdgeMap(
    toy_tree,
    toy_lca_valuation,
)

toy_comparison = pd.DataFrame(
    {
        "source": toy_shaping["edgeMap"]["sources"],
        "target": toy_shaping["edgeMap"]["targets"],
        "shape_space_max_on_contours": toy_shaping["edgeMap"]["values"],
        "hierarchy_lca_projection": toy_lca["values"],
    }
)
display(toy_comparison)

np.testing.assert_array_equal(
    toy_shaping["edgeMap"]["values"],
    np.array([9.0, 0.0, 0.0], dtype=np.float32),
)
np.testing.assert_array_equal(
    toy_lca["values"],
    np.array([9.0, 9.0, 0.0], dtype=np.float32),
)
print("difference between the operators: confirmed")

## Next steps

- Try `COMPACTNESS`, `RECTANGULARITY`, or `BITQUADS_CIRCULARITY`.
- Compare Tree of Shapes, max-tree, and min-tree.
- Use `Minima` when smaller criterion values represent relevant shapes.
- Adjust `MIN_AREA`, `TOP_K`, and `CUT_QUANTILES`.
- Use `HierarchySaliencyMap` for an already monotone original-tree valuation; keep `ShapeSpaceSaliency` for shaping an arbitrary node measure.